# Notebook 9 – Feature Selection

## What is Feature Selection?

Feature selection is the process of selecting the most useful features from a dataset for a Machine Learning model.

The goal is to keep relevant features, remove irrelevant or redundant features, and reduce unnecessary complexity.

### Why Feature Selection?

Feature selection can:

- Reduce the number of input features
- Remove irrelevant information
- Reduce redundant information
- Improve model efficiency
- Reduce training time
- Reduce overfitting
- Improve model interpretability
- Help models focus on useful information

### Basic Workflow

Raw Features
    ↓
Check Relevance
    ↓
Remove Irrelevant Features
    ↓
Remove Redundant Features
    ↓
Apply Feature Selection Method
    ↓
Selected Features
    ↓
Machine Learning Model

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Relevant, Irrelevant, and Redundant Features

### Relevant Features

Relevant features contain useful information related to the prediction problem.

Example:

`Pclass` may be useful for predicting survival.

### Irrelevant Features

Irrelevant features provide little or no useful information for the target.

Example:

`PassengerId` is mainly an identifier and generally does not represent a meaningful passenger characteristic.

### Redundant Features

Redundant features contain information that is already represented by another feature.

For example, if two features are highly correlated, keeping both may provide limited additional information.

### Important Point

Feature selection does not mean simply selecting features that have high correlation with the target.

Different selection methods measure usefulness in different ways.

In [2]:
feature_columns = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

df[feature_columns].describe()

,Pclass,Age,SibSp,Parch,Fare
count,891.000000,714.000000,891.000000,891.000000,891.000000
mean,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.420000,0.000000,0.000000,0.000000
25%,2.000000,20.125000,0.000000,0.000000,7.910400
50%,3.000000,28.000000,0.000000,0.000000,14.454200
75%,3.000000,38.000000,1.000000,0.000000,31.000000
max,3.000000,80.000000,8.000000,6.000000,512.329200


## 1. Filter Methods

Filter methods select features using statistical properties of the data before training a Machine Learning model.

Common filter methods include:

- Variance Threshold
- Correlation-Based Selection
- Mutual Information
- Chi-Square

### Advantages

- Fast
- Simple
- Model-independent
- Suitable for high-dimensional datasets

### Limitations

- Usually evaluate features independently
- May not capture complex interactions between features
- Selected features may not always produce the best model performance

In [3]:
from sklearn.feature_selection import VarianceThreshold

X = df[["Age", "SibSp", "Parch", "Fare", "Pclass"]].copy()
X = X.fillna(X.median())

variance_selector = VarianceThreshold(threshold=0.01)

X_variance = variance_selector.fit_transform(X)

selected_variance_features = X.columns[
    variance_selector.get_support()
]

selected_variance_features.tolist()

['Age', 'SibSp', 'Parch', 'Fare', 'Pclass']

## 2. Variance Threshold

Variance Threshold removes features whose variance is below a selected threshold.

A feature with very low variance has almost the same value for most observations and may provide limited information.

### Feature Selection Decision

Features with variance below the threshold are removed.

### Advantages

- Very simple
- Fast
- Does not require the target variable
- Useful as an initial filtering step

### Limitations

- Low variance does not always mean a feature is irrelevant
- A feature with low variance can still be predictive
- Does not consider relationships with the target

### Final Decision

Use Variance Threshold as an initial filtering method rather than the only feature selection method.

In [4]:
variance_values = X.var()

variance_table = pd.DataFrame({
    "Feature": variance_values.index,
    "Variance": variance_values.values
})

variance_table

,Feature,Variance
0,Age,169.512498
1,SibSp,1.216043
2,Parch,0.649728
3,Fare,2469.436846
4,Pclass,0.699015


## 3. Correlation-Based Selection

Correlation-based selection checks the relationship between numerical features.

Highly correlated features may contain similar information.

For example, if two features have a correlation close to +1 or -1, they may be redundant.

### Advantages

- Easy to understand
- Fast
- Useful for identifying redundant numerical features

### Limitations

- Pearson correlation mainly measures linear relationships
- Correlation does not prove causation
- A feature can have low correlation with another feature but still be useful
- It does not directly measure all types of predictive relationships

In [5]:
correlation_matrix = X.corr()

correlation_matrix

,Age,SibSp,Parch,Fare,Pclass
Age,1.000000,-0.233296,-0.172482,0.096688,-0.339898
SibSp,-0.233296,1.000000,0.414838,0.159651,0.083081
Parch,-0.172482,0.414838,1.000000,0.216225,0.018443
Fare,0.096688,0.159651,0.216225,1.000000,-0.549500
Pclass,-0.339898,0.083081,0.018443,-0.549500,1.000000


## 4. Mutual Information

Mutual Information measures the amount of information one variable provides about another variable.

For feature selection, it can be used to estimate how much information a feature contains about the target.

Unlike simple linear correlation, Mutual Information can capture more general relationships.

### Advantages

- Can capture non-linear relationships
- Useful for classification and regression
- Measures dependency between variables

### Limitations

- More computationally expensive than simple correlation
- Results can depend on estimation
- Does not directly show the direction of the relationship
- Requires careful interpretation

In [6]:
from sklearn.feature_selection import mutual_info_classif

mi_X = df[["Age", "SibSp", "Parch", "Fare", "Pclass"]].copy()
mi_X = mi_X.fillna(mi_X.median())

y = df["Survived"]

mi_scores = mutual_info_classif(
    mi_X,
    y,
    random_state=42
)

mi_table = pd.DataFrame({
    "Feature": mi_X.columns,
    "Mutual_Information": mi_scores
}).sort_values(
    "Mutual_Information",
    ascending=False
)

mi_table

,Feature,Mutual_Information
3,Fare,0.136205
4,Pclass,0.065260
1,SibSp,0.030295
0,Age,0.017137
2,Parch,0.000000


## 5. Chi-Square Selection

Chi-Square is commonly used to measure the relationship between categorical features and a categorical target.

For the Titanic dataset, categorical variables such as `Sex` can be encoded before applying Chi-Square.

### Advantages

- Suitable for categorical features
- Simple statistical method
- Useful for classification problems

### Limitations

- Primarily used with non-negative feature values
- Measures statistical association rather than predictive performance
- Not suitable for continuous negative-valued features without appropriate preprocessing
- A statistically significant relationship does not automatically mean the feature will improve the model

In [7]:
from sklearn.feature_selection import chi2
from sklearn.preprocessing import OneHotEncoder

sex_data = df[["Sex"]].fillna("Unknown")

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

sex_encoded = encoder.fit_transform(sex_data)

chi_scores, p_values = chi2(
    sex_encoded,
    df["Survived"]
)

chi_table = pd.DataFrame({
    "Feature": encoder.get_feature_names_out(["Sex"]),
    "Chi_Square": chi_scores,
    "P_Value": p_values
})

chi_table

,Feature,Chi_Square,P_Value
0,Sex_female,170.348127,6.210585e-39
1,Sex_male,92.702447,6.077838e-22


## 6. Wrapper Methods

Wrapper methods select features by repeatedly training a Machine Learning model using different feature subsets.

Examples include:

- Recursive Feature Elimination (RFE)
- Forward Selection
- Backward Elimination

### Advantages

- Considers feature combinations
- Uses actual model performance
- Can find useful feature subsets

### Limitations

- Computationally expensive
- Can take a long time with many features
- Results depend on the selected model
- Can overfit if validation is not handled correctly

In [8]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe_X = df[["Age", "SibSp", "Parch", "Fare", "Pclass"]].copy()
rfe_X = rfe_X.fillna(rfe_X.median())

model = LogisticRegression(max_iter=1000)

rfe = RFE(
    estimator=model,
    n_features_to_select=3
)

rfe.fit(rfe_X, y)

rfe_table = pd.DataFrame({
    "Feature": rfe_X.columns,
    "Selected": rfe.support_,
    "Ranking": rfe.ranking_
})

rfe_table

,Feature,Selected,Ranking
0,Age,False,2
1,SibSp,True,1
2,Parch,True,1
3,Fare,False,3
4,Pclass,True,1


## 7. Recursive Feature Elimination (RFE)

Recursive Feature Elimination removes the least important features step by step until the required number of features remains.

### Process

1. Train the model using all features.
2. Identify the least important features.
3. Remove them.
4. Train the model again.
5. Continue until the required number of features remains.

### Advantages

- Considers feature combinations
- Uses model information
- Produces a selected subset

### Limitations

- Computationally expensive
- Depends on the selected estimator
- Can be affected by noisy data
- Requires careful validation

In [9]:
selected_rfe_features = rfe_X.columns[rfe.support_].tolist()

selected_rfe_features

['SibSp', 'Parch', 'Pclass']

## 8. Embedded Methods

Embedded methods perform feature selection during the model training process.

Examples include:

- L1/Lasso-based selection
- Tree-based feature importance

### Advantages

- Selection happens during model training
- Usually more efficient than wrapper methods
- Can consider feature usefulness within a model

### Limitations

- Depends on the chosen model
- Different models may select different features
- Some methods can be sensitive to feature scaling

In [10]:
from sklearn.linear_model import LogisticRegression

embedded_X = df[["Age", "SibSp", "Parch", "Fare", "Pclass"]].copy()
embedded_X = embedded_X.fillna(embedded_X.median())

l1_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    max_iter=1000
)

l1_model.fit(embedded_X, y)

l1_coefficients = pd.DataFrame({
    "Feature": embedded_X.columns,
    "Coefficient": l1_model.coef_[0]
})

l1_coefficients

C:\Users\gowda\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\gowda\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,Feature,Coefficient
0,Age,-0.035646
1,SibSp,-0.224639
2,Parch,0.205339
3,Fare,0.005333
4,Pclass,-0.874611


## 9. L1 / Lasso-Based Selection

L1 regularization adds a penalty that can force some model coefficients to become exactly zero.

Features with zero coefficients can therefore be removed from the model.

For classification, L1-regularized Logistic Regression can be used.

For regression, Lasso Regression is commonly used.

### Advantages

- Performs feature selection during model training
- Can produce sparse models
- Useful when there are many features

### Limitations

- Sensitive to feature scale
- Selection can be unstable when features are highly correlated
- Depends on the regularization strength

In [11]:
l1_selected = l1_coefficients[
    l1_coefficients["Coefficient"] != 0
]["Feature"].tolist()

l1_selected

['Age', 'SibSp', 'Parch', 'Fare', 'Pclass']

## 10. Tree-Based Feature Importance

Tree-based models can estimate feature importance based on how useful each feature is for splitting the data.

Random Forest is one example.

### Advantages

- Can capture non-linear relationships
- Can capture feature interactions
- Does not require features to be linearly related
- Useful for ranking features

### Limitations

- Importance can be biased toward certain features
- Correlated features can share or distort importance
- Importance does not necessarily imply causation
- Feature importance should be validated using model performance

In [12]:
from sklearn.ensemble import RandomForestClassifier

tree_X = df[["Age", "SibSp", "Parch", "Fare", "Pclass"]].copy()
tree_X = tree_X.fillna(tree_X.median())

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(tree_X, y)

tree_importance = pd.DataFrame({
    "Feature": tree_X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

tree_importance

,Feature,Importance
3,Fare,0.418731
0,Age,0.383605
4,Pclass,0.088161
1,SibSp,0.062206
2,Parch,0.047299


## Filter vs Wrapper vs Embedded Methods

| Method | Examples | Main Idea | Advantages | Limitations |
|---|---|---|---|---|
| Filter | Variance, Correlation, MI, Chi-Square | Statistical properties | Fast and model-independent | May miss feature combinations |
| Wrapper | RFE | Evaluate feature subsets using a model | Considers combinations | Computationally expensive |
| Embedded | L1, Tree Importance | Selection during training | Efficient and model-aware | Depends on model |

### General Rule

Filter methods are useful for initial feature reduction.

Wrapper methods are useful when a smaller dataset allows more computationally expensive selection.

Embedded methods are useful when the selected Machine Learning model naturally provides feature selection information.

In [13]:
method_summary = pd.DataFrame({
    "Method": [
        "Variance Threshold",
        "Correlation",
        "Mutual Information",
        "Chi-Square",
        "RFE",
        "L1",
        "Tree-Based Importance"
    ],
    "Category": [
        "Filter",
        "Filter",
        "Filter",
        "Filter",
        "Wrapper",
        "Embedded",
        "Embedded"
    ]
})

method_summary

,Method,Category
0,Variance Threshold,Filter
1,Correlation,Filter
2,Mutual Information,Filter
3,Chi-Square,Filter
4,RFE,Wrapper
5,L1,Embedded
6,Tree-Based Importance,Embedded


## Feature Selection Documentation

Feature selection is different from feature creation.

The required documentation below records why each important feature is retained, removed, or requires further analysis.

### Feature: PassengerId

**Feature Name:** PassengerId  
**Source Columns:** PassengerId  
**Logic:** Identifier assigned to each passenger.  
**Reason:** Initially inspected as a possible model input.  
**Business Meaning:** Identifies a passenger record rather than describing passenger behavior.  
**ML Relevance:** Usually provides little meaningful predictive information.  
**Leakage Check:** It does not directly contain the target, but it should not be used as a meaningful predictive feature.  
**Final Decision:** Remove

### Feature: Pclass

**Feature Name:** Pclass  
**Source Columns:** Pclass  
**Logic:** Passenger class value.  
**Reason:** Represents passenger socioeconomic/class information.  
**Business Meaning:** Represents ticket class.  
**ML Relevance:** Can contain useful information for predicting survival.  
**Leakage Check:** Does not directly use the target.  
**Final Decision:** Retain

### Feature: Age

**Feature Name:** Age  
**Source Columns:** Age  
**Logic:** Passenger age.  
**Reason:** Age may influence survival patterns.  
**Business Meaning:** Represents passenger age.  
**ML Relevance:** Can help the model identify age-related patterns.  
**Leakage Check:** Does not use the target.  
**Final Decision:** Retain

### Feature: Fare

**Feature Name:** Fare  
**Source Columns:** Fare  
**Logic:** Passenger fare amount.  
**Reason:** Represents ticket price.  
**Business Meaning:** Represents the amount paid for the ticket.  
**ML Relevance:** Can provide socioeconomic and travel-related information.  
**Leakage Check:** Does not use the target.  
**Final Decision:** Retain

### Feature: SibSp

**Feature Name:** SibSp  
**Source Columns:** SibSp  
**Logic:** Number of siblings/spouses aboard.  
**Reason:** Represents family relationship information.  
**Business Meaning:** Indicates family connections.  
**ML Relevance:** Can help identify family-related survival patterns.  
**Leakage Check:** Does not use the target.  
**Final Decision:** Retain

### Feature: Parch

**Feature Name:** Parch  
**Source Columns:** Parch  
**Logic:** Number of parents/children aboard.  
**Reason:** Represents family relationship information.  
**Business Meaning:** Indicates family connections.  
**ML Relevance:** Can provide information about passenger group structure.  
**Leakage Check:** Does not use the target.  
**Final Decision:** Retain

### Important Note

The final selected feature set should not be decided only from one statistical method.

Feature usefulness should also be evaluated using model performance, domain knowledge, redundancy, interpretability, and leakage risk.

In [14]:
feature_documentation = pd.DataFrame({
    "Feature Name": [
        "PassengerId",
        "Pclass",
        "Age",
        "Fare",
        "SibSp",
        "Parch"
    ],
    "Source Columns": [
        "PassengerId",
        "Pclass",
        "Age",
        "Fare",
        "SibSp",
        "Parch"
    ],
    "Final Decision": [
        "Remove",
        "Retain",
        "Retain",
        "Retain",
        "Retain",
        "Retain"
    ]
})

feature_documentation

,Feature Name,Source Columns,Final Decision
0,PassengerId,PassengerId,Remove
1,Pclass,Pclass,Retain
2,Age,Age,Retain
3,Fare,Fare,Retain
4,SibSp,SibSp,Retain
5,Parch,Parch,Retain


## Feature Selection Workflow

A practical feature selection workflow is:

1. Understand the features and target.
2. Identify irrelevant features.
3. Check low-variance features.
4. Check highly correlated/redundant features.
5. Apply filter methods when appropriate.
6. Apply wrapper methods when computationally practical.
7. Apply embedded methods.
8. Compare model performance.
9. Check interpretability.
10. Check for data leakage.
11. Select the final feature set.

### Important ML Practice

Feature selection should be performed using training data during model development.

The test dataset should not influence which features are selected.

This prevents information from the test set from influencing the model development process.

In [15]:
final_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

selected_data = df[final_features + ["Survived"]]

selected_data.head()

,Pclass,Age,SibSp,Parch,Fare,Survived
0,3,22.0,1,0,7.2500,0
1,1,38.0,1,0,71.2833,1
2,3,26.0,0,0,7.9250,1
3,1,35.0,1,0,53.1000,1
4,3,35.0,0,0,8.0500,0


## Advantages and Limitations of Feature Selection

### Advantages

- Reduces dimensionality
- Reduces computational cost
- Can reduce overfitting
- Can improve model interpretability
- Removes irrelevant and redundant information
- Can improve model training efficiency

### Limitations

- Important features may be removed incorrectly
- Different methods may select different features
- Some methods are computationally expensive
- Selection can depend on the model
- Correlated features can make selection difficult
- Statistical significance does not always mean practical usefulness

### Final Conclusion

Feature selection is the process of identifying the most useful features for a Machine Learning problem.

Filter methods such as Variance Threshold, Correlation, Mutual Information, and Chi-Square are generally fast and useful for initial selection.

Wrapper methods such as RFE evaluate feature subsets using a model but can be computationally expensive.

Embedded methods such as L1 regularization and tree-based feature importance perform selection using the model itself.

The best feature selection approach depends on the dataset, target variable, model, computational resources, and business problem.

In [16]:
print("Final selected features:")
print(final_features)

print("\nFinal dataset shape:")
print(selected_data.shape)

Final selected features:
['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

Final dataset shape:
(891, 6)
